# **MÓDULO 35 - Cross Validation**
## Detecção de incêndios

### Contexto

A detecção precoce de incêndios é um desafio crucial para a preservação ambiental e segurança humana. O monitoramento de variáveis atmosféricas e químicas, como temperatura, umidade, gases e partículas em suspensão, fornece indicadores importantes para prever a ocorrência de incêndios. Nesse cenário, técnicas de ciência de dados podem ser aplicadas para desenvolver modelos preditivos capazes de identificar padrões que antecedem esses eventos críticos. A validação adequada desses modelos garante maior confiabilidade e capacidade de generalização, fundamentais em aplicações de risco elevado como esta.

### 🎯 Objetivo Geral

Aplicar a técnica de **validação cruzada (cross-validation)** para avaliar a performance de um modelo de classificação na previsão da ocorrência de incêndios, utilizando variáveis ambientais coletadas em sensores.


### Objetivos Complementares

* **Pré-processamento pt-1:** Identificar variáveis relevantes, remover colunas não úteis (ex.: índice `Unnamed:0`) e tratar dados inconsistentes.
* **Modelagem:** Implementar modelos de classificação supervisionada (ex.: Regressão Logística, Random Forest, SVM), aplicando validação cruzada para comparar desempenhos.
* **Avaliação:** Medir métricas de performance (acurácia, precisão, recall, F1-score) com base nos folds da validação cruzada, verificando a robustez do modelo.

### 🗄️ Base de Dados

A base de dados contém variáveis ambientais e químicas utilizadas para prever a ocorrência de incêndios, conforme a **Tabela 1**.

| Feature         | Descrição                                             |
| --------------- | ----------------------------------------------------- |
| Unnamed:0       | Índice (não é uma variável útil para o modelo)        |
| UTC             | Tempo em Segundos UTC                                 |
| Temperature\[C] | Temperatura do Ar (°C)                                |
| Humidity\[%]    | Umidade do Ar (%)                                     |
| TVOC\[ppb]      | Compostos Orgânicos Voláteis totais (ppb)             |
| eCO2\[ppm]      | Concentração equivalente de CO₂ (ppm)                 |
| Raw H2          | Hidrogênio molecular bruto, não compensado            |
| Raw Ethanol     | Etanol gasoso bruto                                   |
| Pressure\[hPA]  | Pressão do Ar (hPa)                                   |
| PM1.0           | Material particulado < 1,0 µm                         |
| PM2.5           | Material particulado entre 1,0 µm e 2,5 µm            |
| NC0.5           | Concentração numérica de partículas < 0,5 µm          |
| NC1.0           | Concentração numérica de partículas 0,5–1,0 µm        |
| NC2.5           | Concentração numérica de partículas 1,0–2,5 µm        |
| CNT             | Contador de amostras                                  |
| **Fire Alarm**  | Variável alvo binária: 1 = incêndio, 0 = não incêndio |
**Tabela 1**: Variáveis do dataframe

## 📂 1-Importação de Bibliotecas e Dataload 

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

#sklearn library
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

from sklearn.model_selection import cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Leitura dos dados
df = pd.read_csv("/home/akel/PycharmProjects/EBAC/dados/smoke_detection_iot.csv", delimiter=',')
df.head(10)

,Unnamed: 0,UTC,Temperature[C],Humidity[%],TVOC[ppb],eCO2[ppm],Raw H2,Raw Ethanol,Pressure[hPa],PM1.0,PM2.5,NC0.5,NC1.0,NC2.5,CNT,Fire Alarm
0,0,1654733331,20.000,57.36,0,400,12306,18520,939.735,0.0,0.00,0.0,0.000,0.00,0,0
1,1,1654733332,20.015,56.67,0,400,12345,18651,939.744,0.0,0.00,0.0,0.000,0.00,1,0
2,2,1654733333,20.029,55.96,0,400,12374,18764,939.738,0.0,0.00,0.0,0.000,0.00,2,0
3,3,1654733334,20.044,55.28,0,400,12390,18849,939.736,0.0,0.00,0.0,0.000,0.00,3,0
4,4,1654733335,20.059,54.69,0,400,12403,18921,939.744,0.0,0.00,0.0,0.000,0.00,4,0
5,5,1654733336,20.073,54.12,0,400,12419,18998,939.725,0.0,0.00,0.0,0.000,0.00,5,0
6,6,1654733337,20.088,53.61,0,400,12432,19058,939.738,0.0,0.00,0.0,0.000,0.00,6,0
7,7,1654733338,20.103,53.20,0,400,12439,19114,939.758,0.0,0.00,0.0,0.000,0.00,7,0
8,8,1654733339,20.117,52.81,0,400,12448,19155,939.758,0.0,0.00,0.0,0.000,0.00,8,0
9,9,1654733340,20.132,52.46,0,400,12453,19195,939.756,0.9,3.78,0.0,4.369,2.78,9,0


## 2 - Para essa base, onde você realizará as previsões de fire alarm, qual modelo de machine learning você aplicará? Justifique.

A base de dados utilizada para a detecção de incêndios contém 62.630 observações com 15 variáveis, incluindo atributos contínuos e discretos relacionados a medições ambientais e químicas. Essa composição mista de features é bem manipulada por algoritmos baseados em árvores, como Decision Trees e Random Forest, que são naturalmente capazes de lidar com dados contínuos e discretos ao mesmo tempo, com relações não lineares entre variáveis e interações complexas entre features. outro ponto é não exigir pressupostos sobre a distribuição dos dados ou a normalização rigorosa das variavés. Além disso, o conjunto apresenta baixa dimensionalidade, ou seja, o número de variáveis é muito menor que o número de amostras, o que possibilita treinar modelos ensemble complexos, com baixo riscos de overfitting.

Assim o **Random Forest** apresenta-se como uma das escolhas mais apropriadas. Este algoritmo captura relações não lineares complexas entre variáveis, sendo robusto frente à presença de diferentes tipos de dados, e lida bem com interações e dependências entre features. Outros modelos, como a Regressão Logística ou o Naive Bayes, poderiam ser considerados; entretanto, a regressão logística tem limitações na representação de não linearidades, e o Naive Bayes baseia-se na premissa de independência condicional entre atributos, hipótese que dificilmente se sustenta em contextos ambientais. Dessa forma, as características intrínsecas do conjunto de dados, aliadas à flexibilidade e robustez do Random Forest, sustentam sua escolha como abordagem metodológica apropriada e promissora para a detecção de incêndios, fornecendo uma base sólida para avaliação comparativa de performance usando validação cruzada estratificada.


## 3 - Separe a base em Y e X e já rode a instância do modelo que você utilizará

In [2]:
# Separar a base em X (variaveis) e y (target)
X = df.drop(['Unnamed: 0','UTC','CNT','Fire Alarm'], axis=1)  # remoçao de toda variaveis temporais ou demarcadores de tempo.
y = df['Fire Alarm']

percentagens = (y.value_counts()/len(y)*100).round(2).sort_values(ascending=False)
print("Distribuição em porcentagem:")
print(percentagens)

Distribuição em porcentagem:
Fire Alarm
1    71.46
0    28.54
Name: count, dtype: float64


**Comentários**
Na construção do modelo de classificação para prever a ocorrência de incêndios, foram selecionadas como variáveis explicativas (X) apenas os atributos relacionados a condições ambientais, químicas e físicas do ar, excluindo colunas de controle ou temporais (Unnamed:0, UTC e CNT).

* Unnamed: 0: índice automático da base de dados, sem valor informativo.

* UTC (Tempo em segundos): representa apenas a ordem temporal das medições, mas não traz informação física sobre o incêndio (a não ser que fosse convertido em variáveis derivadas como hora do dia).

* CNT (Contador de amostras): semelhante ao UTC, é apenas um identificador sequencial sem valor explicativo direto.

## 4 - Defina o número de Folds e rode o modelo com a validação cruzada.
**Comentários**\
O problema em questão é uma classificação binária de eventos raros (incêndios), na qual a classe positiva corresponde a aproximadamente **71,46% das observações, enquanto a negativa representa 28,54%**. Esse desbalanceamento exige atenção na escolha da estratégia de validação cruzada. A validação cruzada tradicional, como o KFold, assume que as amostras são independentes e intercambiáveis; entretanto, os dados possuem dependência temporal, de modo que o passado influencia o futuro. Nesse contexto, métodos que respeitam a ordem temporal, como o **TimeSeriesSplit**, permitem simular a previsão em tempo real, treinando com dados passados e testando com dados futuros. Contudo, eles podem gerar folds desbalanceados, prejudicando a avaliação. Por outro lado, quando a prioridade é avaliar a performance comparativa dos modelos, independentemente da ordem temporal, o **StratifiedKFold** pode ser mais adequado, pois preserva a proporção das classes em cada fold, garantindo uma avaliação mais estável e representativa da capacidade do modelo de distinguir entre incêndio e não-incêndio. Assim, a atividade será inicialmente realizada utilizando o KFold tradicional, e em anexo será apresentada a solução com **StratifiedKFold** e **TimeSeriesSplit**


In [3]:
rf_model_1 = RandomForestClassifier()
folds = 12
cross_validation = KFold(n_splits=folds, shuffle=True, random_state=5) 
modelo_1 = cross_val_score(rf_model_1, X, y, cv=cross_validation)

## 5 - Avalie a pontuação de cada modelo e ao final a validação final da média.
**Comentários**\
A função print_cv_scores foi escrita para organizar e apresentar os resultados da validação cruzada de um modelo de machine learning. Ela recebe como entrada os scores retornados pelo cross_validate, as métricas de interesse (accuracy, precision, recall, f1) e um título para o modelo.

In [4]:
def print_cv_scores(scores, scoring,title):
    """Imprime resultados do cross_validate de forma organizada"""
    
    print("📊" +title)
    print("=" * 65)
    
    # Table header
    header = ["Fold"] + [m.upper() for m in scoring] + ["Fit Time", "Score Time"]
    print(f"{header[0]:<6}", end="")
    for h in header[1:]:
        print(f"{h:<12}", end="")
    print()
    
    # Table rows
    n_folds = len(scores['fit_time'])
    for i in range(n_folds):
        print(f"{i+1:<6}", end="")
        for metric in scoring:
            print(f"{scores[f'test_{metric}'][i]:<12.4f}", end="")
        print(f"{scores['fit_time'][i]:<12.4f}{scores['score_time'][i]:<12.4f}")
    
    # Summary
    print("-" * 65)
    print(f"{'MÉDIA':<6}", end="")
    for metric in scoring:
        mean_val = scores[f'test_{metric}'].mean()
        print(f"{mean_val:<12.4f}", end="")
    print(f"{scores['fit_time'].mean():<12.4f}{scores['score_time'].mean():<12.4f}")
    
    print(f"{'STD':<6}", end="")
    for metric in scoring:
        std_val = scores[f'test_{metric}'].std()
        print(f"{std_val:<12.4f}", end="")
    print(f"{scores['fit_time'].std():<12.4f}{scores['score_time'].std():<12.4f}")

    
# métricas
scoring = ['accuracy', 'precision', 'recall', 'f1']
scores = cross_validate(rf_model_1, X, y, cv=cross_validation, scoring=scoring)
print_cv_scores(scores, scoring,"Random florest- KFold")

📊Random florest
Fold  ACCURACY    PRECISION   RECALL      F1          Fit Time    Score Time  
1     1.0000      1.0000      1.0000      1.0000      4.7791      0.0323      
2     0.9998      1.0000      0.9997      0.9999      4.6986      0.0321      
3     1.0000      1.0000      1.0000      1.0000      4.7253      0.0327      
4     1.0000      1.0000      1.0000      1.0000      4.8703      0.0323      
5     1.0000      1.0000      1.0000      1.0000      4.8372      0.0323      
6     1.0000      1.0000      1.0000      1.0000      4.8529      0.0329      
7     1.0000      1.0000      1.0000      1.0000      4.8959      0.0326      
8     1.0000      1.0000      1.0000      1.0000      4.6729      0.0328      
9     1.0000      1.0000      1.0000      1.0000      4.8400      0.0397      
10    0.9998      0.9997      1.0000      0.9999      4.6791      0.0324      
11    1.0000      1.0000      1.0000      1.0000      4.6802      0.0348      
12    1.0000      1.0000      1.0000

**Comentários**\
Os resultados do Random Forest na validação cruzada mostram uma performance extremamente alta e consistente em todas as métricas avaliadas (accuracy, precision, recall e F1), com valores próximos de 1.0 em praticamente todos os folds. O desvio padrão muito baixo indica que o modelo se comporta de forma estável entre os diferentes folds, e o tempo de treino e teste se manteve consistente ao longo da avaliação.

No entanto, essa performance praticamente perfeita pode sugerir possível overfitting, especialmente considerando que algumas features do dataset podem ser altamente informativas ou correlacionadas.  Para validar a robustez do modelo, seria recomendável utilizar estratégias de validação temporal(TimeSeriesSplit- Anexo B), garantindo que a avaliação reflita o desempenho real em cenários de previsão futuros.

## Anexo A- StratifiedKFold

In [5]:
from sklearn.model_selection import  StratifiedKFold

# Modelo
rf_model_2 = RandomForestClassifier(random_state=5)

# Validação cruzada estratificada
cross_validation = StratifiedKFold(n_splits=folds, shuffle=True, random_state=5)

# Métricas
scoring = ['accuracy', 'precision', 'recall', 'f1']

# Avaliação
scores = cross_validate(rf_model_2, X, y, cv=cross_validation, scoring=scoring)
print_cv_scores(scores, scoring, "Random Forest - StratifiedKFold")

📊Random Forest
Fold  ACCURACY    PRECISION   RECALL      F1          Fit Time    Score Time  
1     1.0000      1.0000      1.0000      1.0000      4.8539      0.0334      
2     0.9998      0.9997      1.0000      0.9999      4.9196      0.0328      
3     1.0000      1.0000      1.0000      1.0000      4.8884      0.0334      
4     1.0000      1.0000      1.0000      1.0000      4.9170      0.0332      
5     1.0000      1.0000      1.0000      1.0000      4.9408      0.0332      
6     1.0000      1.0000      1.0000      1.0000      4.8479      0.0335      
7     1.0000      1.0000      1.0000      1.0000      4.8955      0.0328      
8     1.0000      1.0000      1.0000      1.0000      4.9869      0.0332      
9     1.0000      1.0000      1.0000      1.0000      4.8963      0.0329      
10    1.0000      1.0000      1.0000      1.0000      4.9106      0.0334      
11    1.0000      1.0000      1.0000      1.0000      4.8280      0.0329      
12    1.0000      1.0000      1.0000 

**comentário**\
Usando o StratifiedKFold, a performance continua extremamente alta e consistente, similar ao KFold. O modelo mostra excelente capacidade de classificação, e o desvio padrão baixo reforça a estabilidade da avaliação comparativa entre folds. Essa abordagem é mais confiável para comparar modelos em termos de performance, embora ainda não leve em conta a ordem temporal.

## Anexo B- TimeSeriesSplit

In [6]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import make_scorer

# Modelo
rf_model_3 = RandomForestClassifier(random_state=5)

# Validação temporal

tscv = TimeSeriesSplit(n_splits=folds)

# Métricas
scoring = {
    'accuracy': make_scorer(accuracy_score),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall': make_scorer(recall_score, zero_division=0),
    'f1': make_scorer(f1_score, zero_division=0)
}

# Cross-validate
scores = cross_validate(rf_model_3, X, y, cv=tscv, scoring=scoring)

# Exibir resultados
print_cv_scores(scores, scoring, "Random Forest - TimeSeriesSplit")

📊Random Forest - TimeSeriesSplit
Fold  ACCURACY    PRECISION   RECALL      F1          Fit Time    Score Time  
1     0.8296      1.0000      0.8296      0.9068      0.3295      0.0254      
2     0.3388      1.0000      0.3388      0.5061      0.5376      0.0258      
3     0.9996      1.0000      0.9996      0.9998      0.9590      0.0280      
4     1.0000      1.0000      1.0000      1.0000      1.3222      0.0273      
5     0.9485      1.0000      0.8487      0.9182      1.6256      0.0287      
6     0.8157      1.0000      0.8157      0.8985      1.9195      0.0289      
7     1.0000      1.0000      1.0000      1.0000      2.4293      0.0269      
8     1.0000      1.0000      1.0000      1.0000      2.5720      0.0274      
9     1.0000      1.0000      1.0000      1.0000      2.7954      0.0268      
10    0.5124      0.5670      0.8417      0.6776      3.0881      0.0282      
11    0.5568      0.0005      0.5000      0.0009      3.9591      0.0291      
12    1.0000      0

**Comentário**\
O TimeSeriesSplit observamos uma variação muito maior nas métricas entre os folds. Alguns folds apresentam accuracy, recall e F1 muito baixos, refletindo cenários em que os dados futuros não são previsíveis apenas a partir do passado disponível. Isso evidencia que **o modelo não generaliza igualmente bem para todos os intervalos temporais**, mostrando a dificuldade de prever eventos raros em séries temporais. Apesar da média das métricas (accuracy ≈ 0.83, F1 ≈ 0.74) ser menor que nos KFold e StratifiedKFold, esta abordagem fornece uma estimativa mais realista do desempenho em cenários de previsão futuros, simulando o comportamento do modelo em produção.